# UDISE+ Mobile Variant — Simple Central Login Test

**Project owner:** Prashant · UMV Tetahali  
**Purpose:** Prove only the central UDISE login flow on mobile/Colab.  
**Test scope:** Username + Password + manual CAPTCHA → Login → confirm landing page.  
**Open in Colab:** [▶ Launch this test](https://colab.research.google.com/github/zzpsah/udise-automation-suite/blob/main/UDISE_Mobile_Variant_Login_Test_2026-09-23.ipynb)

> This test intentionally does **not** load SDMS, choose modules, select academic session, fetch students, or modify any record. Those steps come only after central login is proven.

> Username/password are entered at runtime only. CAPTCHA is solved manually by the user. The notebook does not bypass CAPTCHA or OTP.


In [ ]:
#@title 1️⃣ Setup browser { display-mode: "form" }
!pip -q install playwright
!playwright install chromium >/dev/null 2>&1

from playwright.async_api import async_playwright
from IPython.display import display, Image
from getpass import getpass
import re, os, time

LOGIN_URL = "https://auth.udiseplus.gov.in/login" #@param {type:"string"}

# Close an older test browser if this cell is rerun.
for obj_name in ("browser", "pw"):
    obj = globals().get(obj_name)
    if obj:
        try:
            if obj_name == "browser": await obj.close()
            else: await obj.stop()
        except Exception:
            pass

pw = await async_playwright().start()
browser = await pw.chromium.launch(headless=True)
context = await browser.new_context(
    viewport={"width": 430, "height": 900},
    user_agent="Mozilla/5.0 (Linux; Android 14; Mobile) AppleWebKit/537.36 Chrome/123 Mobile Safari/537.36"
)
page = await context.new_page()
await page.goto(LOGIN_URL, wait_until="domcontentloaded", timeout=60000)
print("✅ Central login page loaded")
print("Page:", page.url)


In [ ]:
#@title 2️⃣ Login — Username + Password + CAPTCHA { display-mode: "form" }

if "page" not in globals():
    raise RuntimeError("Run Setup browser first.")

# Locate username/password fields by common attributes without hard-coding credentials.
async def first_visible(selectors):
    for sel in selectors:
        loc = page.locator(sel)
        try:
            if await loc.count() and await loc.first.is_visible():
                return loc.first
        except Exception:
            pass
    return None

user_box = await first_visible([
    'input[name*="user" i]', 'input[id*="user" i]',
    'input[placeholder*="user" i]', 'input[type="text"]'
])
pass_box = await first_visible([
    'input[type="password"]', 'input[name*="pass" i]', 'input[id*="pass" i]'
])

if not user_box or not pass_box:
    await page.screenshot(path="/tmp/udise_login_page.png", full_page=True)
    display(Image(filename="/tmp/udise_login_page.png"))
    raise RuntimeError("Could not safely detect username/password fields. Screenshot shown above.")

# Find likely CAPTCHA image and show only that when possible.
captcha = await first_visible([
    'img[src*="captcha" i]', 'img[id*="captcha" i]',
    'img[alt*="captcha" i]', 'img[class*="captcha" i]'
])

if captcha:
    cap_path = "/tmp/udise_captcha.png"
    await captcha.screenshot(path=cap_path)
    print("CAPTCHA:")
    display(Image(filename=cap_path))
else:
    page_path = "/tmp/udise_login_page.png"
    await page.screenshot(path=page_path, full_page=True)
    print("CAPTCHA element was not identified separately. Login-page screenshot:")
    display(Image(filename=page_path))

username = getpass("Username: ").strip()
password = getpass("Password: ")
captcha_text = input("CAPTCHA: ").strip()

await user_box.fill(username)
await pass_box.fill(password)

captcha_box = await first_visible([
    'input[name*="captcha" i]', 'input[id*="captcha" i]',
    'input[placeholder*="captcha" i]'
])
if captcha_box:
    await captcha_box.fill(captcha_text)
elif captcha_text:
    raise RuntimeError("CAPTCHA was entered, but the CAPTCHA input field could not be safely detected.")

# Remove password variable from Python after filling browser field.
password = None

login_btn = await first_visible([
    'button[type="submit"]', 'input[type="submit"]',
    'button:has-text("Login")', 'button:has-text("Sign In")',
    'text=/^login$/i'
])
if not login_btn:
    raise RuntimeError("Login button could not be safely detected.")

before = page.url
try:
    async with page.expect_navigation(wait_until="domcontentloaded", timeout=30000):
        await login_btn.click()
except Exception:
    # Some SPA login pages update without a classic navigation event.
    try:
        await login_btn.click(timeout=3000)
    except Exception:
        pass
    await page.wait_for_timeout(5000)

after = page.url
print("\nLogin result")
print("Before:", before)
print("After :", after)
print("Title :", await page.title())

# Do not print cookies/session values.
still_password = await page.locator('input[type="password"]').count() > 0
if after != before and not still_password:
    print("✅ Login appears to have progressed beyond the central login page.")
    print("STOP HERE for this test. Module/session handling will be added only after this is confirmed working.")
else:
    print("⚠️ Login is not yet confirmed. Check CAPTCHA/credentials or inspect the page message below.")
    # Show a short visible text summary, not HTML/cookies.
    try:
        txt = re.sub(r'\s+', ' ', (await page.locator('body').inner_text())).strip()
        print(txt[:1200])
    except Exception:
        pass


In [ ]:
#@title 3️⃣ Close test browser { display-mode: "form" }
if "browser" in globals():
    await browser.close()
if "pw" in globals():
    await pw.stop()
print("✅ Test browser closed")
